In [2]:
pip install pandas numpy scikit-learn

  Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl (10.0 MB)
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.7 MB 1.9 MB/s eta 0:00:07
   ---- ----------------------------------- 1.3/12.7 MB 3.2 MB/s eta 0:00:04
   ----- ---------------------------------- 1.8/12.7 MB 2.8 MB/s eta 0:00:04
   ------- -------------------------------- 2.4/12.7 MB 2.7 MB/s eta 0:00:04
   -------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

In [5]:
df = pd.read_json('weather.json')
print("✓ Data loaded")
print(f"Shape: {df.shape}")

✓ Data loaded
Shape: (8784, 8)


In [6]:
df = df[['Date/Time', 'Temp_C', 'Rel Hum_%', 'Wind Speed_km/h']].copy()
df.columns = ['timestamp', 'temperature', 'humidity', 'wind_speed']
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed')
df = df.dropna()

print("✓ Data cleaned")
print(f"Total rows: {len(df)}")

✓ Data cleaned
Total rows: 8784


In [7]:
def detect_frozen_sensor(df, column, window=15, threshold=1e-6):
    rolling_std = df[column].rolling(window=window).std()
    frozen_indices = rolling_std[rolling_std < threshold].index.tolist()
    return frozen_indices

In [8]:
def detect_spike_fault(df, column, z_score_threshold=3.5):
    mean = df[column].mean()
    std = df[column].std()
    z_scores = np.abs((df[column] - mean) / std)
    spike_indices = df[z_scores > z_score_threshold].index.tolist()
    return spike_indices

In [9]:
def detect_drift_fault(df, column, window=30, drift_threshold=0.05):
    rolling_mean_recent = df[column].rolling(window=window, min_periods=1).mean()
    rolling_mean_older = rolling_mean_recent.shift(window)
    
    mean_diff = np.abs(rolling_mean_recent - rolling_mean_older)
    relative_drift = mean_diff / (df[column].std() + 1e-6)
    
    drift_indices = df[relative_drift > drift_threshold].index.tolist()
    return drift_indices

In [10]:
def detect_missing_data(df, column):
    missing_indices = df[df[column].isna()].index.tolist()
    return missing_indices

In [11]:
def classify_fault(df, column, index):
    
    fault_info = {
        'index': index,
        'timestamp': df.loc[index, 'timestamp'],
        'value': df.loc[index, column],
        'fault_type': 'Unknown',
        'severity': 'Low'
    }
    
    if pd.isna(df.loc[index, column]):
        fault_info['fault_type'] = 'Missing Data'
        fault_info['severity'] = 'Medium'
        return fault_info
    
    window = min(15, index)
    if window > 0:
        recent_values = df.loc[max(0, index-window):index, column].values
        if len(recent_values) > 1 and np.std(recent_values) < 1e-6:
            fault_info['fault_type'] = 'Frozen Sensor'
            fault_info['severity'] = 'High'
            return fault_info
    
    mean = df[column].mean()
    std = df[column].std()
    z_score = abs((df.loc[index, column] - mean) / (std + 1e-6))
    
    if z_score > 3.5:
        fault_info['fault_type'] = 'Wild Spike'
        fault_info['severity'] = 'High'
        return fault_info
    
    window = min(30, index)
    if window > 5:
        older_values = df.loc[max(0, index-2*window):max(0, index-window), column]
        recent_values = df.loc[max(0, index-window):index, column]
        
        if len(older_values) > 0 and len(recent_values) > 0:
            older_mean = older_values.mean()
            recent_mean = recent_values.mean()
            drift_amount = abs(recent_mean - older_mean)
            
            if drift_amount > 0.1 * std:
                fault_info['fault_type'] = 'Sensor Drift'
                fault_info['severity'] = 'Medium'
                return fault_info
    
    fault_info['fault_type'] = 'No Fault'
    fault_info['severity'] = 'Low'
    return fault_info

In [12]:
def classify_all_faults(df, columns=['temperature', 'humidity', 'wind_speed']):
    faults_detected = []
    
    for column in columns:
        frozen_indices = detect_frozen_sensor(df, column)
        spike_indices = detect_spike_fault(df, column)
        drift_indices = detect_drift_fault(df, column)
        missing_indices = detect_missing_data(df, column)
        
        all_suspect_indices = set(frozen_indices + spike_indices + drift_indices + missing_indices)
        
        for idx in all_suspect_indices:
            fault_classification = classify_fault(df, column, idx)
            fault_classification['sensor'] = column
            faults_detected.append(fault_classification)
    
    faults_df = pd.DataFrame(faults_detected)
    
    if len(faults_df) > 0:
        severity_order = {'Critical': 4, 'High': 3, 'Medium': 2, 'Low': 1}
        faults_df['severity_rank'] = faults_df['severity'].map(severity_order)
        faults_df = faults_df.sort_values(['severity_rank', 'timestamp'], ascending=[False, True])
        faults_df = faults_df.drop('severity_rank', axis=1)
    
    return faults_df

In [13]:
def generate_alerts_summary(faults_df):
    
    if len(faults_df) == 0:
        return {
            'total_faults': 0,
            'critical_count': 0,
            'high_count': 0,
            'medium_count': 0
        }
    
    summary = {
        'total_faults': len(faults_df),
        'critical_count': len(faults_df[faults_df['severity'] == 'Critical']),
        'high_count': len(faults_df[faults_df['severity'] == 'High']),
        'medium_count': len(faults_df[faults_df['severity'] == 'Medium']),
    }
    
    return summary

In [14]:
# Run classification
faults_df = classify_all_faults(df)

# Generate summary
alerts = generate_alerts_summary(faults_df)

# Display results
print("=" * 80)
print("FAULT DETECTION RESULTS")
print("=" * 80)

print(f"\n📊 SUMMARY:")
print(f"   Total Faults Detected: {alerts['total_faults']}")
print(f"   🔴 Critical: {alerts['critical_count']}")
print(f"   🟠 High: {alerts['high_count']}")
print(f"   🟡 Medium: {alerts['medium_count']}")

if len(faults_df) > 0:
    print(f"\n🔝 TOP 20 FAULTS:")
    print(faults_df[['timestamp', 'sensor', 'fault_type', 'severity']].head(20).to_string(index=False))
else:
    print("\n✅ No faults detected!")

print("\n" + "=" * 80)
print("✅ FAULT CLASSIFICATION COMPLETE!")

FAULT DETECTION RESULTS

📊 SUMMARY:
   Total Faults Detected: 23924
   🔴 Critical: 0
   🟠 High: 44
   🟡 Medium: 21793

🔝 TOP 20 FAULTS:
          timestamp     sensor fault_type severity
2012-01-02 12:00:00 wind_speed Wild Spike     High
2012-01-18 01:00:00 wind_speed Wild Spike     High
2012-01-18 02:00:00 wind_speed Wild Spike     High
2012-01-18 03:00:00 wind_speed Wild Spike     High
2012-01-18 04:00:00 wind_speed Wild Spike     High
2012-01-18 05:00:00 wind_speed Wild Spike     High
2012-01-28 23:00:00 wind_speed Wild Spike     High
2012-01-29 00:00:00 wind_speed Wild Spike     High
2012-01-29 01:00:00 wind_speed Wild Spike     High
2012-01-29 02:00:00 wind_speed Wild Spike     High
2012-03-03 13:00:00 wind_speed Wild Spike     High
2012-03-03 14:00:00 wind_speed Wild Spike     High
2012-03-03 15:00:00 wind_speed Wild Spike     High
2012-03-03 16:00:00 wind_speed Wild Spike     High
2012-03-03 17:00:00 wind_speed Wild Spike     High
2012-03-03 18:00:00 wind_speed Wild Spike     Hi